In [57]:
import pandas as pd
import joblib
import explainer
import io
import contextlib
import re
import warnings

warnings.filterwarnings("ignore")

In [ ]:
###########                             ##########
#           Configuration des datasets
###########                             ##########

# Chaque dataset contient :
# - le fichier des règles exclusives
# - les données de test
# - le modèle entraîné
# - le type de format des données

DATASETS = {

    "KDD99": {
        "rules": "../Rules/KDD99_exclusive_rules.csv",
        "X_test": "Datasets/KDD99/X_test.csv",
        "y_test": "Datasets/KDD99/y_test.csv",
        "model": "../Models/KDD99.pkl",
        "type": "csv"
    },

    "BIG15": {
        "rules": "../Rules/BIG15_exclusive_rules.csv",
        "X_test": "Datasets/BIG15/X_test.csv",
        "y_test": "Datasets/BIG15/y_test.csv",
        "model": "../Models/BIG15.pkl",
        "type": "csv"
    },

    "UNSWNB15": {
        "rules": "../Rules/UNSWNB15_exclusive_rules.csv",
        "test": "Datasets/UNSW-NB15/processed/le_test.csv",
        "model": "../Models/UNSW-NB15.pkl",
        "type": "unsw"
    },

    "DoH20": {
        "rules": "../Rules/DOH20_exclusive_rules.csv",
        "X_test": "Datasets/DoH20/X_test.csv",
        "y_test": "Datasets/DoH20/y_test.csv",
        "model": "../Models/DoH20.pkl",
        "type": "csv"
    },

    "CIC2017": {
        "rules": "../Rules/CIC2017_exclusive_rules.csv",
        "X_test": "Datasets/CIC2017/X_test.parquet",
        "y_test": "Datasets/CIC2017/y_test.csv",
        "model": "Models/CIC17.pkl",
        "type": "parquet"
    }
}


In [60]:
###########                             ##########
#            Chargement du dataset
###########                             ##########

def load_dataset(config):
    """
    Charge un dataset selon son format.
    Retourne le dataframe de test avec la colonne label.
    """

    if config["type"] == "csv":

        X_test = pd.read_csv(config["X_test"])
        y_test = pd.read_csv(config["y_test"])

        X_test["label"] = y_test

    elif config["type"] == "parquet":

        X_test = pd.read_parquet(config["X_test"])
        y_test = pd.read_csv(config["y_test"])

        X_test["label"] = y_test

    elif config["type"] == "unsw":

        test = pd.read_csv(config["test"])

        # Suppression des colonnes non utilisées
        X_test = test.drop(["id", "attack_cat"], axis=1)

    else:
        raise ValueError("Format de dataset inconnu")

    return X_test



In [62]:
###########                                        ##########
#           Parser les métriques affichées par CMR
###########                                        ##########

def parse_cmr_output(output_text):

    patterns = {

        "coverage": r"Test set coverage:\s+([\d\.]+)%",
        "f1_coverage": r"Test coverage F1-score:\s+([\d\.]+)",
        "skipped": r"Total Skipped:\s+\d+\s+\(([\d\.]+)%\)",
        "conflicts": r"Due to Conflicts:\s+\d+\s+\(([\d\.]+)%\)",
        "no_rule": r"Due to No Rule Coverage:\s+\d+\s+\(([\d\.]+)%\)",
        "f1_total": r"F1-score \(if using entire dataset\):\s+([\d\.]+)"
    }

    results = {}

    for key, pattern in patterns.items():

        match = re.search(pattern, output_text)

        if match:
            results[key] = float(match.group(1))

    return results


In [ ]:

###########                             ##########
#             Evaluation dataset
###########                             ##########


def evaluate_dataset(dataset_name, config):

    print(f"\n{'='*30}")
    print(f"Evaluation dataset: {dataset_name}")
    print('='*30)

    rules = pd.read_csv(config["rules"])

    model = joblib.load(config["model"])

    X_test = load_dataset(config)

    
    #  Evaluation avec la classe majoritaire

    print("\n--- CLass majoritaire prediction  ---")

    # affichage 
    explainer.apply_exclusive_rules(X_test.copy(), rules)

    # capture des métriques
    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        explainer.apply_exclusive_rules(X_test.copy(), rules)

    metrics_majority = parse_cmr_output(buffer.getvalue())
    
    #  Evaluation avec modèle

    print("\n--- Model prediction ---")

    explainer.apply_exclusive_rules(X_test.copy(), rules, model)

    buffer = io.StringIO()

    with contextlib.redirect_stdout(buffer):
        explainer.apply_exclusive_rules(X_test.copy(), rules, model)

    metrics_model = parse_cmr_output(buffer.getvalue())

    # Résultats

    return {

        "Dataset": dataset_name,
        "Method": "CMR",

        "Coverage (%)": metrics_majority.get("coverage"),
        "F1_coverage": metrics_majority.get("f1_coverage"),

        "Skipped (%)": metrics_majority.get("skipped"),
        "Conflicts (%)": metrics_majority.get("conflicts", 0.0),
        "No_rule (%)": metrics_majority.get("no_rule"),

        "F1_rmaj": metrics_majority.get("f1_total"),
        "F1_rr": metrics_model.get("f1_total")
    }


In [ ]:
###########                             ##########
#                Evaluation globale
###########                             ##########

def run_evaluation():

    results = []

    for dataset_name, config in DATASETS.items():

        res = evaluate_dataset(dataset_name, config)

        results.append(res)

    results_df = pd.DataFrame(results)

    print("\nRésultats finaux :")
    print(results_df)

    results_df.to_csv("rules_results.csv", index=False)

    print("\nEvaluation terminée.")
    print("Fichier généré : rules_results.csv")

    return results_df


In [71]:
###########                             ##########
#                  Execution
###########                             ##########

if __name__ == "__main__":

    run_evaluation()


Evaluating dataset: KDD99

--- Majority class prediction ---


  5%|▌         | 307/5889 [00:00<00:01, 3062.21it/s]

Test set coverage: 69.84%
Test coverage F1-score: 1.00
--------------------------------------
Total Skipped: 8781 (30.16%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 8781 (30.16%)
--------------------------------------
Using Majority class to predict non-covered cases.
F1-score (if using entire dataset): 0.6918


100%|██████████| 5889/5889 [00:01<00:00, 3874.24it/s]



--- Model prediction ---


...

Test set coverage: 93.32%
Test coverage F1-score: 1.00


  0%|          | 0/823366 [00:00<?, ?it/s]

--------------------------------------
Total Skipped: 33710 (6.68%)
 - Due to Conflicts: 0 (0.0%)
 - Due to No Rule Coverage: 33710 (6.68%)
--------------------------------------
Using model to predict non-covered cases.
F1-score (if using entire dataset): 0.9956


100%|██████████| 823366/823366 [09:59<00:00, 1373.16it/s]



Résultats finaux :
    Dataset Method  Coverage (%)  F1_coverage  Skipped (%)  Conflicts (%)  \
0     KDD99    CMR         69.84         1.00        30.16            0.0   
1     BIG15    CMR         94.38         0.99         5.62            0.0   
2  UNSWNB15    CMR         99.77         0.92         0.23            0.0   
3     DoH20    CMR        100.00         1.00         0.00            0.0   
4   CIC2017    CMR         93.32         1.00         6.68            0.0   

   No_rule (%)  F1_rmaj   F1_rr  
0        30.16   0.6918  0.9981  
1         5.62   0.9487  0.9870  
2         0.23   0.9175  0.9172  
3         0.00   1.0000  1.0000  
4         6.68   0.9386  0.9956  

Evaluation terminée.
Fichier généré : rules_results.csv


Interprétation des résultats de l’évaluation des règles : 

L’évaluation a été réalisée sur cinq jeux de données de cybersécurité : **KDD Cup 1999 Dataset, BIG15 Dataset, UNSW‑NB15 Dataset, DoH20 Dataset et CICIDS2017 Dataset.
L’objectif était de mesurer la capacité des règles générées par la méthode CMR (Confusion Matrix Rules) à couvrir les données de test et à produire des prédictions correctes.

L’évaluation repose principalement sur trois indicateurs :

    Coverage (%) : proportion des instances du jeu de test couvertes par les règles.

    F1_coverage : performance des règles uniquement sur les instances couvertes.

    F1 global : performance finale après gestion des instances non couvertes.

Deux stratégies ont été comparées pour les instances non couvertes :

    la classe majoritaire

    les prédictions du modèle

--------Résultats sur KDD99--------------

Pour le dataset KDD99, les règles couvrent 69.84 % des instances du jeu de test. Cela signifie qu’environ 30.16 % des données ne sont couvertes par aucune règle.

Cependant, les règles qui s’appliquent sont extrêmement précises, avec un F1-score de 1.00 sur les instances couvertes. Autrement dit, lorsqu’une règle est activée, la prédiction est quasiment parfaite.

Lorsque les instances non couvertes sont classées avec la classe majoritaire, le F1-score global chute à 0.6918, ce qui montre que cette stratégie est limitée.
En revanche, lorsqu’on utilise les prédictions du modèle, le F1-score global atteint 0.9981, ce qui indique que la combinaison des règles et du modèle permet d’obtenir une performance très élevée.

-------Résultats sur BIG15-----------

Sur le dataset BIG15, la couverture des règles est beaucoup plus élevée, atteignant 94.38 % des données de test.
Le F1-score sur les données couvertes est de 0.99, ce qui montre que les règles restent très fiables.

Seulement 5.62 % des instances ne sont pas couvertes.
Dans ce cas :

    la stratégie classe majoritaire donne un F1-score global de 0.9487

    l’utilisation du modèle améliore ce score à 0.9870

Cela confirme que l’approche hybride règles + modèle est plus efficace.

----------Résultats sur UNSW-NB15--------

Pour le dataset UNSW-NB15, les règles couvrent presque tout le dataset avec une couverture de 99.77 %.
Très peu d’instances restent donc non couvertes (0.23 %).

Le F1-score sur les données couvertes est 0.92, ce qui reste correct mais légèrement inférieur aux autres datasets.
Dans ce cas précis, la stratégie utilisée pour les instances non couvertes (classe majoritaire ou modèle) n’a quasiment aucun impact sur la performance globale, car ces instances sont très peu nombreuses.

--------Résultats sur DoH20-------------

Le dataset DoH20 présente les meilleurs résultats.

Les règles couvrent 100 % des données, ce qui signifie qu’aucune instance ne nécessite de prédiction supplémentaire.
Le F1-score est également de 1.00, indiquant que les règles permettent de classer parfaitement les données de test.

Dans ce cas, l’utilisation du modèle ou de la classe majoritaire ne change rien puisque les règles suffisent à elles seules.

---------Résultats sur CIC2017--------------

Le dataset CIC2017 contient un volume de données très important (plus de 800 000 instances).
Malgré cela, les règles couvrent 93.32 % des données, ce qui est un très bon niveau de couverture.

Le F1-score sur les données couvertes est de 1.00, ce qui signifie que les règles sont extrêmement précises lorsqu’elles s’appliquent.

Pour les 6.68 % d’instances non couvertes :

    la classe majoritaire donne un F1-score global de 0.9386

    l’utilisation du modèle améliore fortement la performance jusqu’à 0.9956

=========Analyse globale===========

De manière générale, les résultats montrent que les règles extraites par la méthode CMR sont très précises. Sur la plupart des datasets, le F1-score sur les données couvertes est proche de 1, ce qui indique que les règles identifiées capturent des motifs très discriminants dans les données.

La couverture varie selon les datasets :

    faible à moyenne pour KDD99 (≈70 %)

    très élevée pour BIG15 et CIC2017 (≈93–94 %)

    presque totale pour UNSW-NB15

    totale pour DoH20


Enfin, l’expérience montre que l’utilisation du modèle pour prédire les instances non couvertes permet d’obtenir de meilleures performances globales que l’utilisation de la classe majoritaire. Cela confirme l’intérêt d’une approche hybride combinant règles interprétables et modèle prédictif.